# 🔬 Programa: Machine Learning para la Toma de Decisiones
## Módulo 1 — Fundamentos Estadísticos: Pruebas de Hipótesis
### Dataset: Agricultural Yield Prediction (Kaggle)
**10 000 parcelas agrícolas | 46 variables | 4 cultivos | 4 regiones**

---

> Este notebook aplica los **6 pasos oficiales** de la comprobación de hipótesis
>
> ⚠️ **Nota sobre unidad de análisis:** los datos son registros de parcelas agrupadas por `Crop_Type × Season × Region × Year`. Para pruebas sobre medias (Casos 1 y 2) se usa el dataset **agregado** `df_agg` (n≈1 199). Para conteos categoriales (Caso 3) se usa `df` individual (n=10 000).

---

## 🗺️ Estructura del notebook

| Caso | Pregunta de investigación | Prueba estadística |
|------|--------------------------|-------------------|
| **Caso 1** | ¿El rendimiento en la estación *Kharif* es distinto al de *Rabi*? | Prueba **t** de dos muestras independientes |

---

### Los 6 pasos que aplicaremos en cada caso

```
PASO 1 → Plantear hipótesis  (H₀ y Hₐ)
PASO 2 → Recoger y preparar los datos
PASO 3 → Elegir la prueba estadística adecuada
PASO 4 → Calcular el estadístico de prueba y el valor-p
PASO 5 → Tomar la decisión  (comparar p vs α)
PASO 6 → Presentar conclusiones
```

In [ ]:
# ── Librerías ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import (
    ttest_ind, f_oneway, chi2_contingency,
    shapiro, levene, norm
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (11, 4.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})
sns.set_palette("husl")
np.random.seed(42)

print("✅ Librerías listas")
print(f"   NumPy {np.__version__}  |  Pandas {pd.__version__}")

---
## 📦 Descripción del Dataset

**Fuente:** [Agricultural Yield Prediction — Kaggle](https://www.kaggle.com/datasets/samuelotiattakorah/agriculture-crop-yield)

Este dataset contiene registros de **10 000 parcelas agrícolas** con información sobre:

| Grupo | Variables |
|-------|-----------|
| Clima | `Temperature`, `Humidity`, `Rainfall`, `Wind_Speed`, `Solar_Radiation` |
| Suelo | `Soil_Type`, `pH`, `EC`, `OC`, `N`, `P`, `K`, `Ca`, `Mg`, `Bulk_Density` |
| Cultivo | `Crop_Type`, `Season`, `Region`, `Planting_Date`, `Harvest_Date` |
| Manejo | `Fertilizer_Type`, `Pesticide_Usage`, `Irrigation_Frequency` |
| **Objetivo** | **`Yield`** — rendimiento en toneladas/hectárea (1.0 – 10.0) |

In [ ]:
# ── Cargar dataset ────────────────────────────────────────────────────────
df = pd.read_csv('Agri_yield_prediction.csv')

print(f"Filas × Columnas : {df.shape}")
print(f"Variables numéricas : {df.select_dtypes('number').shape[1]}")
print(f"Variables categóricas: {df.select_dtypes('object').shape[1]}")
print(f"\nValores nulos: {df.isnull().sum().sum()}")
print(f"\n--- Vista previa ---")
df.head(3)

In [ ]:
# ── Estadísticas descriptivas generales ───────────────────────────────────
resumen = df.groupby('Season')['Yield'].agg(['count','mean','std','median']).round(3)
resumen.columns = ['n', 'Media', 'Desv. Est.', 'Mediana']
print("Rendimiento (Yield) por Estación:")
print(resumen)


print()
resumen2 = df.groupby('Soil_Type')['Yield'].agg(['count','mean','std']).round(3)
resumen2.columns = ['n', 'Media', 'Desv. Est.']
print("Rendimiento por Tipo de Suelo:")
print(resumen2)

print()
ct = pd.crosstab(df['Fertilizer_Type'], df['Crop_Type'])
print("Tabla cruzada: Fertilizante × Cultivo (conteos):")
print(ct)

---
## ⚠️ Definición precisa de la Unidad de Análisis

Antes de cualquier prueba estadística, debemos responder: **¿qué constituye una observación independiente?**

### Estructura real del dataset

Cada fila del CSV representa **una parcela agrícola individual**, pero todas las parcelas comparten un contexto definido por:

| Variable de contexto | Tipo | Valores únicos |
|----------------------|------|---------------|
| `Crop_Type` | Categórica | 4 (Maize, Rice, Soybean, Wheat) |
| `Season` | Categórica | 3 (Kharif, Rabi, Zaid) |
| `Region` | Categórica | 4 (East, North, South, West) |
| `Year` | Numérica | 25 (2000–2024) |

### El problema: pseudoreplicación

Si usamos las 10 000 filas crudas como si fueran 10 000 observaciones independientes:
- Inflamos artificialmente el tamaño de muestra (~8× más)
- Cualquier efecto mínimo se vuelve "significativo" → **conclusiones engañosas**
- Esto se llama **pseudoreplicación** (Hurlbert, 1984)

### Solución: agregar al nivel correcto

Cada combinación `Crop_Type × Season × Region × Year` es la **unidad de análisis** real.

```python
df_agg = df.groupby(['Crop_Type','Season','Region','Year'])['Yield'].mean().reset_index()
# Resultado: ~1 199 observaciones verdaderamente independientes (en lugar de 10 000)
```

> Con esta corrección, los tests estadísticos miden el efecto real, no el ruido amplificado.

In [ ]:
# ── Crear el dataset agregado (unidad de análisis correcta) ───────────────
df_agg = (df.groupby(['Crop_Type','Season','Region','Year'])['Yield']
            .mean()
            .reset_index())

print("─" * 60)
print("Construcción de la Unidad de Análisis correcta")
print("─" * 60)
print(f"\n  Dataset original    : {df.shape[0]:>6} filas  (parcelas individuales)")
print(f"  Dataset agregado    : {df_agg.shape[0]:>6} filas  (combinaciones Crop×Season×Region×Year)")
print(f"  Factor de reducción : {df.shape[0]/df_agg.shape[0]:.1f}×")

print(f"\n  Combinaciones por Season (dataset agregado):")
print(df_agg['Season'].value_counts().to_string())

print(f"\n  Combinaciones por Crop_Type (dataset agregado):")
print(df_agg['Crop_Type'].value_counts().to_string())

print(f"\n  Combinaciones por Region (dataset agregado):")
print(df_agg['Region'].value_counts().to_string())

print(f"\n  Estadísticas de Yield agregado:")
print(df_agg['Yield'].describe().round(3).to_string())

# Verificación: ¿existen combinaciones faltantes?
total_esperado = df['Crop_Type'].nunique() * df['Season'].nunique() * \
                 df['Region'].nunique() * df['Year'].nunique()
print(f"\n  Combinaciones esperadas: {total_esperado}  |  Presentes: {df_agg.shape[0]}"
      f"  |  Faltantes: {total_esperado - df_agg.shape[0]}")
print("\n✅ df_agg listo — se usará en Casos 1 y 2")

---
---
# 🧪 CASO 1 — Prueba t de dos muestras independientes

## Pregunta de investigación
> **¿Es el rendimiento agrícola promedio de la estación Kharif significativamente diferente al de la estación Rabi?**

*Los agricultores y planificadores del sector necesitan saber si la estación de cultivo impacta el rendimiento para diseñar mejor las políticas de subsidio y almacenamiento.*

---

## ✅ PASO 1 — Plantear las hipótesis

$$H_0: \mu_{Kharif} = \mu_{Rabi} \quad \text{(el rendimiento medio es igual en ambas estaciones)}$$

$$H_a: \mu_{Kharif} \neq \mu_{Rabi} \quad \text{(el rendimiento medio es diferente — prueba bilateral)}$$

| | Descripción |
|--|-------------|
| **H₀** | No hay diferencia en el rendimiento promedio entre Kharif y Rabi |
| **Hₐ** | Sí existe diferencia (puede ser en cualquier dirección) |
| **Nivel de significancia** | α = 0.05 |

> 💡 **Unidad de análisis:** cada observación es el rendimiento medio de una combinación `Crop_Type × Region × Year` bajo esa estación. Usamos `df_agg` (~400 obs/estación), no las ~3 400 filas crudas por estación.

---
## ✅ PASO 2 — Recoger y preparar los datos

**Población de interés:** combinaciones Crop_Type × Region × Year del dataset  
**Unidad de análisis:** rendimiento medio anual por crop/región/estación  
**Variable de respuesta:** `Yield` (rendimiento medio en t/ha en `df_agg`)  
**Variable de agrupación:** `Season` — tomamos `Kharif` y `Rabi`  
**n esperado:** ≈ 400 por estación (4 crops × 4 regiones × 25 años)

In [ ]:
df_agg.groupby('Season').count()

In [ ]:
# ── PASO 2: Preparar los datos (usando df_agg) ────────────────────────────
kharif = df_agg[df_agg['Season'] == 'Kharif']['Yield'].dropna()
rabi   = df_agg[df_agg['Season'] == 'Rabi'  ]['Yield'].dropna()

print("─" * 60)
print("PASO 2 — Resumen de los datos (unidad de análisis correcta)")
print("─" * 60)
print(f"  Dataset usado   : df_agg  (Crop_Type × Season × Region × Year)")
print(f"  Estación Kharif : n={len(kharif):>5} | media={kharif.mean():.3f} | std={kharif.std():.3f}")
print(f"  Estación Rabi   : n={len(rabi):>5} | media={rabi.mean():.3f} | std={rabi.std():.3f}")
print(f"\n  Diferencia observada de medias : {kharif.mean() - rabi.mean():.4f} t/ha")
print(f"\n  (Comparar con datos crudos: n_Kharif≈3401, n_Rabi≈3319 → pseudoreplicación)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for datos, nombre, color in [
        (kharif, 'Kharif', '#4C72B0'),
        (rabi,   'Rabi',   '#C44E52')]:
    axes[0].hist(datos, bins=30, alpha=0.55, color=color,
                 label=f'{nombre} (n={len(datos)})', edgecolor='white', density=True)
    axes[0].axvline(datos.mean(), color=color, lw=2.5, linestyle='--')

axes[0].set_title('Distribución del Rendimiento MEDIO\nKharif vs Rabi  (df_agg)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Yield medio (t/ha)'); axes[0].set_ylabel('Densidad'); axes[0].legend()

sns.boxplot(data=df_agg[df_agg['Season'].isin(['Kharif','Rabi'])],
            x='Season', y='Yield', palette=['#4C72B0','#C44E52'],
            width=0.45, ax=axes[1])
axes[1].set_title('Boxplot del Rendimiento MEDIO\nKharif vs Rabi  (df_agg)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Estación'); axes[1].set_ylabel('Yield medio (t/ha)')

plt.tight_layout()
plt.show()

---
## ✅ PASO 3 — Elegir la prueba estadística adecuada

Usamos el árbol de decisión:

```
Tipo de datos  → Continuo (Yield en t/ha)
Grupos         → 2 grupos independientes (Kharif ≠ Rabi)
Distribución   → Verificaremos normalidad (Shapiro-Wilk)
Varianzas      → Verificaremos homocedasticidad (Levene)
```

**Prueba elegida: t de Welch (dos muestras independientes)**  
La variante de Welch es más robusta cuando los tamaños de muestra o varianzas difieren, y es la opción por defecto en SciPy.

$$t = \frac{\bar{x}_{Kharif} - \bar{x}_{Rabi}}{\sqrt{\dfrac{s^2_{Kharif}}{n_{Kharif}} + \dfrac{s^2_{Rabi}}{n_{Rabi}}}}$$

In [ ]:
# ── PASO 3: Verificar supuestos antes de aplicar la prueba t ──────────────
print("─" * 52)
print("PASO 3 — Verificación de supuestos")
print("─" * 52)

# 3a. Normalidad — Shapiro-Wilk (sobre muestra de 300 por velocidad)
muestras_check = {
    'Kharif': kharif.sample(300, random_state=42),
    'Rabi':   rabi.sample(300, random_state=42),
}
print("\n  Test de Shapiro-Wilk (normalidad, muestra=300):")
for nombre, datos in muestras_check.items():
    W, p_sw = shapiro(datos)
    estado = "✅ Normal" if p_sw > 0.05 else "⚠️  No normal (pero n grande → TCL aplica)"
    print(f"    {nombre}: W={W:.4f}, p={p_sw:.4f}  → {estado}")

# 3b. Homocedasticidad — Levene
lev_stat, lev_p = levene(kharif, rabi)
equal_var = lev_p > 0.05
print(f"\n  Test de Levene (homocedasticidad):")
print(f"    F={lev_stat:.4f}, p={lev_p:.4f}  → {'Varianzas iguales ✅' if equal_var else 'Varianzas distintas → usamos Welch ✅'}")

print(f"\n  ► Decisión: Prueba t de Welch  (equal_var=False)")
print(f"    Con n>3000 por grupo, el TCL garantiza robustez aunque no haya normalidad perfecta.")

---
## ✅ PASO 4 — Calcular el estadístico de prueba y el valor-p

Ejecutamos la prueba t de Welch y calculamos manualmente el **intervalo de confianza del 95%** y el **tamaño del efecto (Cohen's d)**.

In [ ]:
# ── PASO 4: Calcular estadístico t y valor-p ─────────────────────────────
alpha = 0.05

t_stat, p_value = ttest_ind(kharif, rabi, equal_var=False, alternative='two-sided')

# Intervalo de confianza del 95% para la diferencia de medias
diff   = kharif.mean() - rabi.mean()
se     = np.sqrt(kharif.var(ddof=1)/len(kharif) + rabi.var(ddof=1)/len(rabi))
ci_low = diff - 1.96 * se
ci_up  = diff + 1.96 * se

# Cohen's d (tamaño del efecto)
sp = np.sqrt(((len(kharif)-1)*kharif.std()**2 + (len(rabi)-1)*rabi.std()**2) /
             (len(kharif) + len(rabi) - 2))
cohens_d = diff / sp

def interpretar_d(d):
    d = abs(d)
    if d < 0.2: return "Negligible"
    elif d < 0.5: return "Pequeño"
    elif d < 0.8: return "Mediano"
    else: return "Grande"

print("─" * 52)
print("PASO 4 — Resultados del cálculo")
print("─" * 52)
print(f"  Media Kharif       : {kharif.mean():.4f} t/ha")
print(f"  Media Rabi         : {rabi.mean():.4f} t/ha")
print(f"  Diferencia (K - R) : {diff:.4f} t/ha")
print(f"\n  t-estadístico      : {t_stat:.4f}")
print(f"  Valor-p (bilateral): {p_value:.6f}")
print(f"\n  IC 95% diferencia  : [{ci_low:.4f}, {ci_up:.4f}]")
print(f"  Cohen's d          : {cohens_d:.4f}  → {interpretar_d(cohens_d)}")

# ── Visualización de la distribución nula y el estadístico observado ───────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: distribución t bajo H₀
from scipy.stats import t as t_dist
dof = (kharif.var(ddof=1)/len(kharif) + rabi.var(ddof=1)/len(rabi))**2 / \
      ((kharif.var(ddof=1)/len(kharif))**2/(len(kharif)-1) +
       (rabi.var(ddof=1)/len(rabi))**2/(len(rabi)-1))

x_t  = np.linspace(-5, 5, 500)
y_t  = t_dist.pdf(x_t, df=dof)
t_crit = t_dist.ppf(1 - alpha/2, df=dof)

ax1.plot(x_t, y_t, 'steelblue', lw=2.5, label=f'Distribución t (gl≈{dof:.0f})')
ax1.fill_between(x_t, y_t, where=(x_t <= -t_crit), color='crimson', alpha=0.45, label=f'Zona de rechazo α/2')
ax1.fill_between(x_t, y_t, where=(x_t >=  t_crit), color='crimson', alpha=0.45)
ax1.axvline(t_stat, color='darkorange', lw=2.5, linestyle='--',
            label=f't observado = {t_stat:.2f}')
ax1.axvline(-t_crit, color='gray', lw=1.2, linestyle=':')
ax1.axvline( t_crit, color='gray', lw=1.2, linestyle=':')
ax1.text(-t_crit, max(y_t)*0.85, f'-{t_crit:.2f}', ha='center', color='gray', fontsize=9)
ax1.text( t_crit, max(y_t)*0.85, f'+{t_crit:.2f}', ha='center', color='gray', fontsize=9)
ax1.set_title(f'Distribución bajo H₀\np = {p_value:.6f}', fontsize=12, fontweight='bold')
ax1.set_xlabel('Estadístico t'); ax1.legend(fontsize=9)

# Panel 2: intervalo de confianza de la diferencia
ax2.errorbar(x=0, y=diff, yerr=[[diff - ci_low], [ci_up - diff]],
             fmt='o', color='darkorange', capsize=10, capthick=2.5,
             markersize=10, lw=2.5, label=f'Diferencia = {diff:.4f}\nIC 95%: [{ci_low:.3f}, {ci_up:.3f}]')
ax2.axhline(0, color='crimson', linestyle='--', lw=2, label='H₀: diferencia = 0')
ax2.set_xlim(-0.5, 0.5); ax2.set_xticks([])
ax2.set_title('Intervalo de Confianza (95%)\nde la diferencia de medias', fontsize=12, fontweight='bold')
ax2.set_ylabel('Diferencia de medias (t/ha)'); ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## ✅ PASO 5 — Tomar la decisión

Comparamos el valor-p con el nivel de significancia predeterminado **α = 0.05**:

| Condición | Regla de decisión |
|-----------|-------------------|
| p-value ≤ α | **Rechazar H₀** → evidencia estadística de diferencia |
| p-value > α | **No rechazar H₀** → evidencia insuficiente |

In [ ]:
# ── PASO 5: Decisión ──────────────────────────────────────────────────────
print("─" * 52)
print("PASO 5 — Decisión estadística")
print("─" * 52)
print(f"\n  p-value   = {p_value:.6f}")
print(f"  α         = {alpha}")
print(f"  p ≤ α ?   → {'SÍ' if p_value <= alpha else 'NO'}")
print()
if p_value <= alpha:
    print("  ✅ DECISIÓN: RECHAZAMOS H₀")
    print("  Los datos proporcionan evidencia estadística suficiente")
    print("  para concluir que los rendimientos medios de Kharif y Rabi")
    print("  son significativamente diferentes (α = 0.05).")
else:
    print("  ❌ DECISIÓN: NO rechazamos H₀")
    print("  No hay evidencia suficiente para concluir que los rendimientos")
    print("  medios de Kharif y Rabi son distintos.")

---
## ✅ PASO 6 — Presentar las conclusiones

### Reporte estadístico formal

> Se realizó una prueba t de Welch para comparar el rendimiento agrícola medio (Yield, t/ha) entre las estaciones Kharif (n = 3 401) y Rabi (n = 3 319).

### Resultados a reportar siempre

| Métrica | Valor |
|---------|-------|
| t-estadístico | Ver celda anterior |
| Valor-p (bilateral) | Ver celda anterior |
| IC 95% de la diferencia | Ver celda anterior |
| Tamaño del efecto (Cohen's d) | Ver celda anterior |

### Importancia práctica vs. importancia estadística

- **¿Es significativo?** → Responde el valor-p
- **¿Es importante?** → Responde Cohen's d y el IC

> ⚠️ Un resultado **estadísticamente significativo** con Cohen's d pequeño puede no tener **relevancia práctica** para la toma de decisiones agrícolas. Siempre reporta ambos.

In [ ]:
# ── PASO 6: Reporte completo del Caso 1 ──────────────────────────────────
print("=" * 62)
print("INFORME FINAL — CASO 1")
print("Pregunta: ¿Difieren los rendimientos de Kharif y Rabi?")
print("=" * 62)
print(f"\n  Contexto:")
print(f"    Dataset: Agricultural Yield Prediction (Kaggle, n=10 000)")
print(f"    Variable: Yield (t/ha)  |  Grupos: Kharif vs Rabi")

print(f"\n  Hipótesis:")
print(f"    H₀: μ_Kharif = μ_Rabi")
print(f"    Hₐ: μ_Kharif ≠ μ_Rabi  (bilateral, α = {alpha})")

print(f"\n  Datos:")
print(f"    Kharif → n={len(kharif)}, x̄={kharif.mean():.3f}, s={kharif.std():.3f}")
print(f"    Rabi   → n={len(rabi)},   x̄={rabi.mean():.3f}, s={rabi.std():.3f}")

print(f"\n  Prueba aplicada: t de Welch (dos muestras independientes)")
print(f"    t = {t_stat:.4f}")
print(f"    p = {p_value:.6f}")
print(f"    IC 95% diferencia: [{ci_low:.4f}, {ci_up:.4f}]")
print(f"    Cohen's d = {cohens_d:.4f}  ({interpretar_d(cohens_d)})")

print(f"\n  Decisión: p {'≤' if p_value <= alpha else '>'} α = {alpha}  "
      f"→ {'Rechazamos H₀ ✅' if p_value <= alpha else 'No rechazamos H₀ ❌'}")

print(f"\n  Conclusión en lenguaje sencillo:")
if p_value <= alpha:
    print(f"    Existe diferencia estadísticamente significativa entre el")
    print(f"    rendimiento de la estación Kharif ({kharif.mean():.3f} t/ha) y la")
    print(f"    estación Rabi ({rabi.mean():.3f} t/ha). Sin embargo, el tamaño del")
    print(f"    efecto es {interpretar_d(cohens_d).lower()} (d={cohens_d:.3f}), lo que sugiere")
    print(f"    que la diferencia práctica puede ser limitada.")
else:
    print(f"    No hay evidencia de que la estación afecte el rendimiento.")

print(f"\n  Implicaciones para el sector agrícola:")
print(f"    Los planificadores deben considerar la estación al proyectar")
print(f"    rendimientos, aunque otros factores (suelo, cultivo) pueden")
print(f"    tener mayor impacto práctico (ver Casos 2 y 3).")
print("=" * 62)